# Play Call Predictor (MDP Architecture)

**What this notebook does:**
1. Loads NFL play-by-play data from `nflreadpy` (2015–2025, ~1.5M plays)
2. Engineers game-state features: down, distance bucket, yard line zone, score diff bucket, quarter, time remaining bucket, shotgun, goal-to-go
3. Labels each play as one of 4 classes: **run**, **pass**, **punt**, **field_goal**
4. Trains a lightweight MLP classifier with **aggressively rebalanced class weights** to fix run-heavy bias
5. Validates predicted probabilities match real NFL play-call rates
6. Saves weights + encoder artifacts

**Key fixes vs v1:**
- Pass class weight boosted 2× on top of inverse-frequency weighting -> fixes 64% run bias
- Passing situations (2nd/3rd & long, shotgun) oversampled 1.5x so model sees realistic distributions
- Calibration check validates against known NFL rates (1st&10 ~45% run, 3rd&8 ~80% pass)
- Temperature parameter saved for inference sharpening
- **Playoff mode**: BOTH teams play at full intensity... model trained on all high-stakes situations equally

**Before running:** Switch runtime to T4 GPU (Free) or A100 (Pro).

**Output artifacts:**
- `play_call_model.pt`
- `play_call_label_encoder.pkl`
- `play_call_feature_meta.json`
- `play_call_config.json` (includes `inference_temperature`)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Switch to T4 GPU runtime before proceeding.')

## 1: Install dependencies

In [ ]:
%%capture
!pip install nflreadpy scikit-learn pandas numpy torch --quiet

## 2: Load play-by-play data

In [ ]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

SEASONS = list(range(2015, 2026))
print(f'Loading PBP for seasons: {SEASONS}')

pbp_raw = nfl.load_pbp(SEASONS).to_pandas()
print(f'Loaded: {pbp_raw.shape[0]:,} plays x {pbp_raw.shape[1]} columns')

## 3: Filter to meaningful scrimmage plays

In [ ]:
KEEP_TYPES = {'run', 'pass', 'punt', 'field_goal'}

df = pbp_raw[
    pbp_raw['play_type'].isin(KEEP_TYPES) &
    pbp_raw['down'].notna() &
    pbp_raw['ydstogo'].notna() &
    pbp_raw['yardline_100'].notna() &
    pbp_raw['score_differential'].notna() &
    pbp_raw['qtr'].notna() &
    pbp_raw['game_seconds_remaining'].notna()
].copy()

df['shotgun'] = df['shotgun'].fillna(0).astype(int)
df['goal_to_go'] = df['goal_to_go'].fillna(0).astype(int)

print(f'After filtering: {len(df):,} plays')
print('\nPlay type distribution:')
dist = df['play_type'].value_counts(normalize=True) * 100
for pt, pct in dist.items():
    print(f'  {pt:<12} {pct:.1f}%')

## 4: Feature engineering

In [ ]:
def dist_bucket(x):
    if x <= 2: return 0
    elif x <= 6: return 1
    elif x <= 10: return 2
    else: return 3

def yard_zone(x):
    if x >= 80: return 0
    elif x >= 60: return 1
    elif x >= 40: return 2
    elif x >= 20: return 3
    elif x >= 5: return 4
    else: return 5

def score_bucket(x):
    if x <= -17: return 0
    elif x <= -7: return 1
    elif x <= 6: return 2
    elif x <= 16: return 3
    else: return 4

def time_bucket(row):
    secs = row['game_seconds_remaining']
    qtr = row['qtr']
    if secs <= 120 and qtr >= 4: return 0
    elif secs <= 480 and qtr == 4: return 1
    elif qtr == 3 or (qtr == 4 and secs > 480): return 2
    elif qtr == 2: return 3
    else: return 4

print('Applying feature buckets...')
df['feat_down'] = df['down'].astype(int) - 1
df['feat_dist'] = df['ydstogo'].apply(dist_bucket)
df['feat_zone'] = df['yardline_100'].apply(yard_zone)
df['feat_score'] = df['score_differential'].apply(score_bucket)
df['feat_qtr'] = (df['qtr'].clip(1, 5) - 1).astype(int)
df['feat_time'] = df.apply(time_bucket, axis=1)
df['feat_half_end'] = (
    ((df['qtr'] == 2) & (df['game_seconds_remaining'] <= 1860)) |
    ((df['qtr'] >= 4) & (df['game_seconds_remaining'] <= 120))
).astype(int)
df['feat_trailing'] = (df['score_differential'] < 0).astype(int)
df['feat_shotgun'] = df['shotgun']
df['feat_goal_to_go'] = df['goal_to_go']

FEATURE_COLS = [
    'feat_down', 'feat_dist', 'feat_zone', 'feat_score',
    'feat_qtr', 'feat_time', 'feat_half_end', 'feat_trailing',
    'feat_shotgun', 'feat_goal_to_go',
]

FEAT_CARDINALITY = {
    'feat_down': 4,
    'feat_dist': 4,
    'feat_zone': 6,
    'feat_score': 5,
    'feat_qtr': 5,
    'feat_time': 5,
    'feat_half_end': 2,
    'feat_trailing': 2,
    'feat_shotgun': 2,
    'feat_goal_to_go': 2,
}

print('Feature engineering done.')
print(df[FEATURE_COLS].head(3))

## 5: Encode labels

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pickle, json, os

le = LabelEncoder()
df['label'] = le.fit_transform(df['play_type'])

print('Classes:', list(le.classes_))
print('Label mapping:', {k: int(v) for k, v in zip(le.classes_, le.transform(le.classes_))})

N_CLASSES = len(le.classes_)
print(f'N_CLASSES: {N_CLASSES}')

## 6: Oversample passing situations to fix run-heavy bias

The raw dataset is ~42% run / ~47% pass, but the model over-predicts runs on neutral downs
because run plays cluster on 1st down (by far the most common down). We oversample
2nd & long, 3rd & medium/long, and shotgun passing situations by 50% so the model
sees the correct pass rate in those contexts.

**Playoff note:** High-stakes situations (Q4 clutch, large deficits) are included equally
in training — both the favored and underdog team play intensely, so the model learns
realistic play-calling under pressure for ANY team, not just top-tier offenses.

In [ ]:
from sklearn.model_selection import train_test_split

passing_situation = (
    ((df['down'] >= 2) & (df['ydstogo'] >= 7) & (df['play_type'].isin(['run','pass']))) |
    ((df['down'] == 3) & (df['ydstogo'] >= 4) & (df['play_type'].isin(['run','pass']))) |
    ((df['shotgun'] == 1) & (df['play_type'].isin(['run','pass'])))
)

df_passing = df[passing_situation]
print(f'Passing situation plays: {len(df_passing):,}')
print('Distribution within passing situations:')
for pt, pct in (df_passing['play_type'].value_counts(normalize=True)*100).items():
    print(f'  {pt:<12} {pct:.1f}%')

df_aug = pd.concat([df, df_passing.sample(frac=0.5, random_state=42)], ignore_index=True)
print(f'\nAugmented dataset: {len(df_aug):,}')
print('Augmented distribution:')
for pt, pct in (df_aug['play_type'].value_counts(normalize=True)*100).items():
    print(f'{pt:<12} {pct:.1f}%')

X = df_aug[FEATURE_COLS].values.astype(np.int64)
y = df_aug['label'].values.astype(np.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)

print(f'\nTrain: {len(X_train):,}  Val: {len(X_val):,}')
print('Train class distribution:')
for cls, lbl in zip(le.classes_, range(N_CLASSES)):
    pct = (y_train == lbl).sum() / len(y_train) * 100
    print(f'  {cls:<12} {pct:.1f}%')

## 7: Define the MLP model with temperature support

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

EMB_DIM = 8
HIDDEN = 512
DROPOUT = 0.3

class PlayCallMLP(nn.Module):
    def __init__(self, feat_cardinality, emb_dim, hidden, n_classes):
        super().__init__()
        self.feat_names = list(feat_cardinality.keys())
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, emb_dim) for card in feat_cardinality.values()
        ])
        in_dim = len(feat_cardinality) * emb_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), 
            nn.LayerNorm(hidden), 
            nn.SiLU(), 
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden), 
            nn.LayerNorm(hidden), 
            nn.SiLU(), 
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden), 
            nn.LayerNorm(hidden), 
            nn.SiLU(), 
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden // 2), 
            nn.LayerNorm(hidden // 2), 
            nn.SiLU(),
            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, x):
        embs = [self.embeddings[i](x[:, i]) for i in range(len(self.feat_names))]
        return self.net(torch.cat(embs, dim=-1))

    def predict_proba(self, x, temperature=1.0):
        return torch.softmax(self.forward(x) / temperature, dim=-1)

model = PlayCallMLP(FEAT_CARDINALITY, EMB_DIM, HIDDEN, N_CLASSES).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 8: Train with boosted pass class weight

Inverse-frequency weights are computed first, then the **pass** weight is multiplied
by `PASS_BOOST=2.0` to aggressively counteract the run-heavy prediction bias.
Re-normalize after boosting so total weight stays proportional.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 80
BATCH_SIZE = 4096
LR = 3e-3
PASS_BOOST = 2.0

X_tr_t = torch.LongTensor(X_train).to(DEVICE)
y_tr_t = torch.LongTensor(y_train).to(DEVICE)
X_vl_t = torch.LongTensor(X_val).to(DEVICE)
y_vl_t = torch.LongTensor(y_val).to(DEVICE)

train_ds = TensorDataset(X_tr_t, y_tr_t)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

class_counts = np.bincount(y_train)
class_weights = 1.0 / class_counts.astype(float)
class_weights = class_weights / class_weights.sum() * N_CLASSES

pass_idx = list(le.classes_).index('pass')
class_weights[pass_idx] *= PASS_BOOST
class_weights = class_weights / class_weights.sum() * N_CLASSES

class_weights_t = torch.FloatTensor(class_weights).to(DEVICE)
print('Final class weights:')
for cls, w in zip(le.classes_, class_weights):
    print(f'{cls:<12} {w:.3f}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(train_dl), epochs=EPOCHS
)

best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_dl:
        optimizer.zero_grad()
        loss = F.cross_entropy(model(xb), yb, weight=class_weights_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_vl_t)
        val_loss = F.cross_entropy(val_logits, y_vl_t, weight=class_weights_t).item()
        val_acc = (val_logits.argmax(-1) == y_vl_t).float().mean().item()

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | train {total_loss/len(train_dl):.4f} '
              f'| val {val_loss:.4f} | acc {val_acc*100:.2f}%')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/tmp/play_call_model_best.pt')

print(f'\nBest val accuracy: {best_val_acc*100:.2f}%')

## 9: Temperature sweep + calibration check

Target rates from real NFL data:
- 1st & 10, neutral: ~45% run, ~55% pass
- 3rd & 8+, trailing: ~15% run, ~85% pass
- 4th & short: ~30% run, ~30% pass, ~25% punt
- Shotgun: ~25% run, ~75% pass

Find the temperature where 1st&10 neutral hits those targets.

In [ ]:
model.load_state_dict(torch.load('/tmp/play_call_model_best.pt', map_location=DEVICE))
model.eval()

with torch.no_grad():
    preds = model(X_vl_t).argmax(-1).cpu().numpy()

print('Per-class accuracy:')
for i, cls in enumerate(le.classes_):
    mask = y_val == i
    if mask.sum() == 0: continue
    print(f'{cls:<12} acc={(preds[mask]==i).mean()*100:.1f}%  n={mask.sum():,}')

print('\n--- Temperature sweep on 1st&10 own25 tied Q1 (target: ~45% run ~55% pass) ---')
x_ref = torch.LongTensor([[0, 2, 1, 2, 0, 4, 0, 0, 0, 0]]).to(DEVICE)
for temp in [0.5, 0.7, 0.8, 0.9, 1.0, 1.2]:
    with torch.no_grad():
        p = model.predict_proba(x_ref, temperature=temp).cpu().numpy()[0]
    print(f'temp={temp:.1f}  ' + '  '.join(f'{c}:{p2*100:.0f}%' for c, p2 in zip(le.classes_, p)))

SCENARIOS = [
    ('1st&10, own25, tied, Q1 [~45% run ~55% pass]', [0, 2, 1, 2, 0, 4, 0, 0, 0, 0]),
    ('2nd&3, own40, tied, Q2 [~55% run ~45% pass]', [1, 0, 1, 2, 1, 3, 0, 0, 0, 0]),
    ('3rd&8, own30, trail, Q4 [~15% run ~85% pass]', [2, 3, 1, 1, 3, 1, 0, 1, 1, 0]),
    ('4th&2, opp38, close, Q4 [~30% run ~30% pass]', [3, 0, 3, 2, 3, 1, 0, 0, 0, 0]),
    ('4th&5, opp18, tied, Q4 [~60% FG]', [3, 1, 4, 2, 3, 1, 0, 0, 0, 0]),
    ('3rd&10, own10, deficit Q4 [~10% run ~90% pass]', [2, 3, 0, 0, 3, 1, 0, 1, 1, 0]),
    ('1st&goal, opp3, tied Q4 GTG [~60% run]', [0, 0, 5, 2, 3, 1, 0, 0, 0, 1]),
    ('1st&10, own30, shotgun Q2 [~25% run ~75% pass]', [0, 2, 1, 2, 1, 3, 0, 0, 1, 0]),
]

DISPLAY_TEMP = 0.8
print(f'\nCalibration at temperature={DISPLAY_TEMP} (classes: {list(le.classes_)}):')
print(f'{"Scenario":<48} ' + '  '.join(f'{c:<9}' for c in le.classes_))
print('-' * 88)
for label, feats in SCENARIOS:
    x = torch.LongTensor([feats]).to(DEVICE)
    with torch.no_grad():
        p = model.predict_proba(x, temperature=DISPLAY_TEMP).cpu().numpy()[0]
    print(f'{label:<48} ' + '  '.join(f'{v*100:6.1f}%  ' for v in p))

## 10: Set inference temperature and save config

In [ ]:
INFERENCE_TEMPERATURE = 0.8

x_ref = torch.LongTensor([[0, 2, 1, 2, 0, 4, 0, 0, 0, 0]]).to(DEVICE)
with torch.no_grad():
    p = model.predict_proba(x_ref, temperature=INFERENCE_TEMPERATURE).cpu().numpy()[0]
print(f'1st&10 neutral at temp={INFERENCE_TEMPERATURE}:')
for cls, prob in zip(le.classes_, p):
    print(f'{cls:<12} {prob*100:.1f}%')

## 11: Save all artifacts

In [ ]:
import shutil

SAVE_DIR = '/tmp/play_call_weights'
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.copy('/tmp/play_call_model_best.pt', f'{SAVE_DIR}/play_call_model.pt')

with open(f'{SAVE_DIR}/play_call_label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

feature_meta = {
    'feature_cols': FEATURE_COLS,
    'feat_cardinality': FEAT_CARDINALITY,
    'classes': list(le.classes_),
}
with open(f'{SAVE_DIR}/play_call_feature_meta.json', 'w') as f:
    json.dump(feature_meta, f, indent=2)

model_config = {
    'emb_dim': EMB_DIM,
    'hidden': HIDDEN,
    'dropout': DROPOUT,
    'n_classes': N_CLASSES,
    'n_features': len(FEATURE_COLS),
    'inference_temperature': INFERENCE_TEMPERATURE,
    'pass_boost': PASS_BOOST,
    'best_val_acc': best_val_acc,
    'training_seasons': SEASONS,
}
with open(f'{SAVE_DIR}/play_call_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print(f'Saved to {SAVE_DIR}:')
for fname in sorted(os.listdir(SAVE_DIR)):
    print(f'{fname:<45} {os.path.getsize(f"{SAVE_DIR}/{fname}")/1024:.1f} KB')

## 12: Download weights zip

In [ ]:
from google.colab import files

zip_path = '/tmp/play_call_weights_export'
shutil.make_archive(zip_path, 'zip', SAVE_DIR)
files.download(f'{zip_path}.zip')
print('Download started. Extract into backend/python_backend/play_call_weights/')

## Appendix: Quick inference test

In [ ]:
import json as _json
with open(f'{SAVE_DIR}/play_call_config.json') as f:
    cfg = _json.load(f)

m2 = PlayCallMLP(FEAT_CARDINALITY, EMB_DIM, HIDDEN, N_CLASSES).to(DEVICE)
m2.load_state_dict(torch.load(f'{SAVE_DIR}/play_call_model.pt', map_location=DEVICE))
m2.eval()
temp = cfg['inference_temperature']

with open(f'{SAVE_DIR}/play_call_label_encoder.pkl', 'rb') as f:
    le2 = pickle.load(f)

print(f'Cold-load OK. Temperature: {temp}')

for label, feats in [
    ('1st&10 neutral [expect ~45% run, ~55% pass]', [0, 2, 1, 2, 0, 4, 0, 0, 0, 0]),
    ('3rd&8 trailing Q4 shotgun [expect ~85% pass]', [2, 3, 1, 1, 3, 1, 0, 1, 1, 0]),
]:
    x = torch.LongTensor([feats]).to(DEVICE)
    with torch.no_grad():
        p = m2.predict_proba(x, temperature=temp).cpu().numpy()[0]
    print(f'\n{label}:')
    for cls, prob in zip(le2.classes_, p):
        print(f'{cls:<12} {prob*100:5.1f}%  {"#"*int(prob*40)}')